In [1]:
import cv2
import mediapipe as mp
import numpy as np

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.5
)

def extract_landmarks(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None
    
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    if not result.multi_hand_landmarks:
        return None  # No hand detected

    lm = result.multi_hand_landmarks[0].landmark

    # Collect 63 values: x1,y1,z1, x2,y2,z2, ...
    features = []
    for point in lm:
        features.extend([point.x, point.y, point.z])

    return features


In [2]:
import os
import pandas as pd

dataset_path = r"C:\Users\ADMIN\Downloads\merged_ASL_dataset"
save_csv_path = r"C:\Users\ADMIN\Downloads\mediapipe_landmarks_ASL.csv"

data = []
labels = []

for category in os.listdir(dataset_path):
    category_path = os.path.join(dataset_path, category)
    if not os.path.isdir(category_path):
        continue

    print("Processing category:", category)

    for file in os.listdir(category_path):
        if file.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")):
            img_path = os.path.join(category_path, file)

            features = extract_landmarks(img_path)  # Mediapipe function
            if features is not None:
                data.append(features)
                labels.append(category)

df = pd.DataFrame(data)
df["label"] = labels
df.to_csv(save_csv_path, index=False)

print("Saved landmark CSV:", save_csv_path)
print("Total extracted:", len(df))


Processing category: A
Processing category: B
Processing category: C
Processing category: D
Processing category: del
Processing category: E
Processing category: F
Processing category: G
Processing category: H
Processing category: I
Processing category: J
Processing category: K
Processing category: L
Processing category: M
Processing category: N
Processing category: nothing
Processing category: O
Processing category: P
Processing category: Q
Processing category: R
Processing category: S
Processing category: space
Processing category: T
Processing category: U
Processing category: V
Processing category: W
Processing category: X
Processing category: Y
Processing category: Z
Saved landmark CSV: C:\Users\ADMIN\Downloads\mediapipe_landmarks_ASL.csv
Total extracted: 36936


In [3]:
import pandas as pd
from sklearn.utils import shuffle

# Input CSV path (your Mediapipe landmark file)
csv_input = r"C:\Users\ADMIN\Downloads\mediapipe_landmarks_ASL.csv"

# Output shuffled CSV path
csv_output = r"C:\Users\ADMIN\Downloads\mediapipe_landmarks_ASL_shuffled.csv"

# Load CSV
df = pd.read_csv(csv_input)

# Shuffle rows randomly
df = shuffle(df, random_state=42)  # random_state makes it reproducible

# Save shuffled CSV
df.to_csv(csv_output, index=False)

print(" Shuffled CSV saved successfully at:")
print(csv_output)
print("Total rows:", len(df))


 Shuffled CSV saved successfully at:
C:\Users\ADMIN\Downloads\mediapipe_landmarks_ASL_shuffled.csv
Total rows: 36936


In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pickle

# Initialize Mediapipe Hand Detection
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.5
)


In [ ]:
def extract_landmarks_from_image(image_path):
    img = cv2.imread(image_path)

    if img is None:
        print("❌ Cannot read image:", image_path)
        return None

    # Convert BGR -> RGB (important for Mediapipe)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    result = hands.process(img_rgb)

    # No hand found
    if not result.multi_hand_landmarks:
        print("⚠ No hand detected in image.")
        return None

    # 21 landmarks × (x,y,z) = 63 features
    features = []
    for lm in result.multi_hand_landmarks[0].landmark:
        features.extend([lm.x, lm.y, lm.z])

    return np.array(features).reshape(1, -1)


In [ ]:
# Load only SVM model
svm_model = pickle.load(open("svm_mediapipe.pkl_1", "rb"))
print("✔ SVM Model Loaded")

def predict_with_svm(image_path):
    features = extract_landmarks_from_image(image_path)

    if features is None:
        return

    pred = svm_model.predict(features)[0]
    print("\n=== SVM Prediction ===")
    print("Predicted Label:", pred)

# Test your external image
test_img = r"C:\Users\ADMIN\Downloads\W2-test.jpg"
predict_with_svm(test_img)
